In [7]:
# Step 1 — imports

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    log_loss,
    roc_auc_score
)
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

print("Imports loaded.")

Imports loaded.


In [8]:
# Step 2 — load Stage 1 engineered dataset

DATA_DIR = Path("../data")

stage1 = pd.read_csv(
    DATA_DIR / "stage1_engineered_features.csv"
)

print("Shape:", stage1.shape)

print("\nColumns:")
print(stage1.columns.tolist())

print("\nTarget distribution:")
print(
    stage1["missed_game_target"]
    .value_counts()
    .sort_index()
)

Shape: (24, 11)

Columns:
['player', 'gsis_id', 'season', 'start_week', 'team', 'report_out_or_doubtful', 'prior_hamstring_episodes', 'pre3_total_actions_avg', 'pre3_games_used', 'missed_game_target', 'pre3_workload_missing']

Target distribution:
missed_game_target
0     9
1    15
Name: count, dtype: int64


In [9]:
# Step 3 — establish naive baseline

y = stage1["missed_game_target"].astype(int)

baseline_probability = y.mean()
baseline_predictions = np.ones(len(y), dtype=int)
baseline_probabilities = np.repeat(
    baseline_probability,
    len(y)
)

baseline_accuracy = accuracy_score(
    y,
    baseline_predictions
)

baseline_balanced_accuracy = balanced_accuracy_score(
    y,
    baseline_predictions
)

baseline_brier = brier_score_loss(
    y,
    baseline_probabilities
)

baseline_log_loss = log_loss(
    y,
    baseline_probabilities
)

print(f"Historical missed-game rate: {baseline_probability:.3f}")
print(f"Historical missed-game rate: {baseline_probability:.1%}")

print("\nMajority-class baseline:")
print(f"Accuracy:          {baseline_accuracy:.3f}")
print(f"Balanced accuracy: {baseline_balanced_accuracy:.3f}")

print("\nConstant-probability baseline:")
print(f"Brier score:       {baseline_brier:.3f}")
print(f"Log loss:          {baseline_log_loss:.3f}")

print("\nConfusion matrix:")
print(confusion_matrix(y, baseline_predictions))

Historical missed-game rate: 0.625
Historical missed-game rate: 62.5%

Majority-class baseline:
Accuracy:          0.625
Balanced accuracy: 0.500

Constant-probability baseline:
Brier score:       0.234
Log loss:          0.662

Confusion matrix:
[[ 0  9]
 [ 0 15]]


In [10]:
# Step 4 — define Stage 1 predictors and target

feature_cols = [
    "report_out_or_doubtful",
    "prior_hamstring_episodes",
    "pre3_total_actions_avg"
]

X = stage1[feature_cols].copy()
y = stage1["missed_game_target"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nPredictors:")
print(X.head())

print("\nMissing values:")
print(X.isna().sum())

print("\nTarget:")
print(y.value_counts().sort_index())

X shape: (24, 3)
y shape: (24,)

Predictors:
   report_out_or_doubtful  prior_hamstring_episodes  pre3_total_actions_avg
0                       0                         0               30.000000
1                       1                         1               25.000000
2                       0                         0               32.333333
3                       0                         0               41.000000
4                       0                         0               38.333333

Missing values:
report_out_or_doubtful      0
prior_hamstring_episodes    0
pre3_total_actions_avg      1
dtype: int64

Target:
missed_game_target
0     9
1    15
Name: count, dtype: int64


In [11]:
# Step 5 — build regularized logistic regression pipeline

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "logistic",
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="liblinear",
            max_iter=1000,
            random_state=42
        )
    )
])

print(model)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('logistic',
                 LogisticRegression(max_iter=1000, penalty='l2',
                                    random_state=42, solver='liblinear'))])


In [12]:
# Step 6 — leave-one-out cross-validated predictions

loo = LeaveOneOut()

loocv_probabilities = cross_val_predict(
    model,
    X,
    y,
    cv=loo,
    method="predict_proba"
)[:, 1]

loocv_predictions = (
    loocv_probabilities >= 0.50
).astype(int)

results = stage1[
    ["player", "season", "start_week", "missed_game_target"]
].copy()

results["predicted_probability"] = loocv_probabilities
results["predicted_class"] = loocv_predictions

print(
    results
    .sort_values("predicted_probability", ascending=False)
    .to_string(index=False)
)

         player  season  start_week  missed_game_target  predicted_probability  predicted_class
     Brad Smith    2011          17                   1               0.991422                1
 Johnny Manziel    2014          17                   1               0.964748                1
   Michael Vick    2015           7                   1               0.945920                1
     Jay Cutler    2015           3                   1               0.914207                1
 Blaine Gabbert    2013           6                   1               0.871009                1
   Dak Prescott    2024          10                   1               0.854477                1
   Daniel Jones    2020          13                   1               0.813659                1
Jacoby Brissett    2023          17                   1               0.809683                1
   Kyler Murray    2019          17                   0               0.747403                1
Taylor Heinicke    2023          12     

/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/min

In [13]:
# Step 7 — evaluate LOOCV model performance

model_accuracy = accuracy_score(
    y,
    loocv_predictions
)

model_balanced_accuracy = balanced_accuracy_score(
    y,
    loocv_predictions
)

model_auc = roc_auc_score(
    y,
    loocv_probabilities
)

model_brier = brier_score_loss(
    y,
    loocv_probabilities
)

model_log_loss = log_loss(
    y,
    loocv_probabilities
)

model_cm = confusion_matrix(
    y,
    loocv_predictions
)

print("LOOCV logistic regression:")
print(f"Accuracy:          {model_accuracy:.3f}")
print(f"Balanced accuracy: {model_balanced_accuracy:.3f}")
print(f"ROC-AUC:           {model_auc:.3f}")
print(f"Brier score:       {model_brier:.3f}")
print(f"Log loss:          {model_log_loss:.3f}")

print("\nConfusion matrix:")
print(model_cm)

LOOCV logistic regression:
Accuracy:          0.667
Balanced accuracy: 0.667
ROC-AUC:           0.719
Brier score:       0.215
Log loss:          0.631

Confusion matrix:
[[ 6  3]
 [ 5 10]]


In [14]:
# Step 8 — compare model against historical baseline

comparison = pd.DataFrame({
    "metric": [
        "Accuracy",
        "Balanced Accuracy",
        "Brier Score",
        "Log Loss"
    ],
    "baseline": [
        baseline_accuracy,
        baseline_balanced_accuracy,
        baseline_brier,
        baseline_log_loss
    ],
    "loocv_logistic": [
        model_accuracy,
        model_balanced_accuracy,
        model_brier,
        model_log_loss
    ]
})

comparison["difference"] = (
    comparison["loocv_logistic"]
    - comparison["baseline"]
)

print(comparison.round(3).to_string(index=False))

print(
    f"\nLOOCV ROC-AUC: {model_auc:.3f} "
    "(no directly comparable ROC-AUC for the constant baseline)"
)

           metric  baseline  loocv_logistic  difference
         Accuracy     0.625           0.667       0.042
Balanced Accuracy     0.500           0.667       0.167
      Brier Score     0.234           0.215      -0.019
         Log Loss     0.662           0.631      -0.031

LOOCV ROC-AUC: 0.719 (no directly comparable ROC-AUC for the constant baseline)


In [15]:
# Step 9 — compare progressively larger Stage 1 models

feature_sets = {
    "report_only": [
        "report_out_or_doubtful"
    ],

    "report_plus_history": [
        "report_out_or_doubtful",
        "prior_hamstring_episodes"
    ],

    "full_model": [
        "report_out_or_doubtful",
        "prior_hamstring_episodes",
        "pre3_total_actions_avg"
    ]
}

model_comparisons = []

for model_name, features in feature_sets.items():

    X_temp = stage1[features].copy()

    temp_prob = cross_val_predict(
        model,
        X_temp,
        y,
        cv=LeaveOneOut(),
        method="predict_proba"
    )[:, 1]

    temp_pred = (temp_prob >= 0.50).astype(int)

    model_comparisons.append({
        "model": model_name,
        "n_features": len(features),
        "accuracy": accuracy_score(y, temp_pred),
        "balanced_accuracy": balanced_accuracy_score(y, temp_pred),
        "roc_auc": roc_auc_score(y, temp_prob),
        "brier": brier_score_loss(y, temp_prob),
        "log_loss": log_loss(y, temp_prob)
    })

model_comparison_df = pd.DataFrame(model_comparisons)

print(
    model_comparison_df
    .round(3)
    .to_string(index=False)
)

/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/min

              model  n_features  accuracy  balanced_accuracy  roc_auc  brier  log_loss
        report_only           1     0.292              0.233    0.467  0.199     0.558
report_plus_history           2     0.583              0.622    0.467  0.222     0.607
         full_model           3     0.667              0.667    0.719  0.215     0.631


/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/min

In [16]:
# Step 10 — inspect LOOCV classification errors

results["correct"] = (
    results["predicted_class"]
    == results["missed_game_target"]
)

errors = (
    results.loc[
        ~results["correct"]
    ]
    .sort_values("predicted_probability", ascending=False)
)

print("Correct predictions:", results["correct"].sum())
print("Incorrect predictions:", (~results["correct"]).sum())

print("\nMisclassified episodes:")
print(
    errors.to_string(index=False)
)

Correct predictions: 16
Incorrect predictions: 8

Misclassified episodes:
        player  season  start_week  missed_game_target  predicted_probability  predicted_class  correct
  Kyler Murray    2019          17                   0               0.747403                1    False
 David Garrard    2010          15                   0               0.525344                1    False
      Joe Webb    2010          14                   0               0.503909                1    False
  Michael Vick    2013           6                   1               0.489916                0    False
Russell Wilson    2022           7                   1               0.408603                0    False
 Justin Fields    2024           8                   1               0.357910                0    False
  Jeff Driskel    2019          13                   1               0.251370                0    False
  Kyler Murray    2022          10                   1               0.053714                0

In [17]:
# Step 11 — regularization sensitivity analysis

C_values = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0, 10.0]

sensitivity_results = []

for C in C_values:

    sensitivity_model = Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="median")
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "logistic",
            LogisticRegression(
                penalty="l2",
                C=C,
                solver="liblinear",
                max_iter=1000,
                random_state=42
            )
        )
    ])

    probs = cross_val_predict(
        sensitivity_model,
        X,
        y,
        cv=LeaveOneOut(),
        method="predict_proba"
    )[:, 1]

    preds = (probs >= 0.50).astype(int)

    sensitivity_results.append({
        "C": C,
        "accuracy": accuracy_score(y, preds),
        "balanced_accuracy": balanced_accuracy_score(y, preds),
        "roc_auc": roc_auc_score(y, probs),
        "brier": brier_score_loss(y, probs),
        "log_loss": log_loss(y, probs)
    })

sensitivity_df = pd.DataFrame(sensitivity_results)

print(
    sensitivity_df
    .round(3)
    .to_string(index=False)
)

/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/min

    C  accuracy  balanced_accuracy  roc_auc  brier  log_loss
 0.05     0.708              0.744    0.719  0.220     0.632
 0.10     0.708              0.744    0.711  0.212     0.611
 0.25     0.667              0.689    0.704  0.208     0.598
 0.50     0.708              0.722    0.704  0.210     0.606
 1.00     0.667              0.667    0.719  0.215     0.631
 2.00     0.667              0.667    0.719  0.220     0.673
 5.00     0.708              0.700    0.733  0.225     0.742
10.00     0.708              0.700    0.733  0.228     0.790


/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [18]:
# Step 12 — inspect standardized full-model coefficients

final_model = model.fit(X, y)

logistic = final_model.named_steps["logistic"]

coefficient_table = pd.DataFrame({
    "feature": X.columns,
    "coefficient": logistic.coef_[0],
    "odds_ratio": np.exp(logistic.coef_[0])
})

coefficient_table["abs_coefficient"] = (
    coefficient_table["coefficient"].abs()
)

coefficient_table = (
    coefficient_table
    .sort_values("abs_coefficient", ascending=False)
    .drop(columns="abs_coefficient")
)

print(
    coefficient_table
    .round(3)
    .to_string(index=False)
)

print(
    "\nIntercept:",
    round(logistic.intercept_[0], 3)
)

                 feature  coefficient  odds_ratio
  report_out_or_doubtful        0.905       2.472
  pre3_total_actions_avg       -0.742       0.476
prior_hamstring_episodes        0.201       1.222

Intercept: 0.672


/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


In [20]:
#can the model generalize to a quarter it has never seen before?

In [21]:
# Step 13 — leave-one-quarterback-out validation

from sklearn.model_selection import LeaveOneGroupOut

groups = stage1["player"]

logo = LeaveOneGroupOut()

lopo_probabilities = cross_val_predict(
    model,
    X,
    y,
    groups=groups,
    cv=logo,
    method="predict_proba"
)[:, 1]

lopo_predictions = (
    lopo_probabilities >= 0.50
).astype(int)

lopo_accuracy = accuracy_score(
    y,
    lopo_predictions
)

lopo_balanced_accuracy = balanced_accuracy_score(
    y,
    lopo_predictions
)

lopo_auc = roc_auc_score(
    y,
    lopo_probabilities
)

lopo_brier = brier_score_loss(
    y,
    lopo_probabilities
)

lopo_log_loss = log_loss(
    y,
    lopo_probabilities
)

print("Leave-one-player-out validation:")
print(f"Accuracy:          {lopo_accuracy:.3f}")
print(f"Balanced accuracy: {lopo_balanced_accuracy:.3f}")
print(f"ROC-AUC:           {lopo_auc:.3f}")
print(f"Brier score:       {lopo_brier:.3f}")
print(f"Log loss:          {lopo_log_loss:.3f}")

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y,
        lopo_predictions
    )
)

Leave-one-player-out validation:
Accuracy:          0.667
Balanced accuracy: 0.667
ROC-AUC:           0.711
Brier score:       0.196
Log loss:          0.556

Confusion matrix:
[[ 6  3]
 [ 5 10]]


/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/miniconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/sofiamartinez/min

In [22]:
# Step 14 — compare baseline, LOOCV, and leave-one-player-out

validation_comparison = pd.DataFrame({
    "method": [
        "Historical baseline",
        "LOOCV logistic",
        "Leave-one-player-out logistic"
    ],
    "accuracy": [
        baseline_accuracy,
        model_accuracy,
        lopo_accuracy
    ],
    "balanced_accuracy": [
        baseline_balanced_accuracy,
        model_balanced_accuracy,
        lopo_balanced_accuracy
    ],
    "roc_auc": [
        np.nan,
        model_auc,
        lopo_auc
    ],
    "brier": [
        baseline_brier,
        model_brier,
        lopo_brier
    ],
    "log_loss": [
        baseline_log_loss,
        model_log_loss,
        lopo_log_loss
    ]
})

print(
    validation_comparison
    .round(3)
    .to_string(index=False)
)

                       method  accuracy  balanced_accuracy  roc_auc  brier  log_loss
          Historical baseline     0.625              0.500      NaN  0.234     0.662
               LOOCV logistic     0.667              0.667    0.719  0.215     0.631
Leave-one-player-out logistic     0.667              0.667    0.711  0.196     0.556


In [23]:
# Step 15 — save final Stage 1 validation results

validation_path = DATA_DIR / "stage1_validation_results.csv"

validation_comparison.to_csv(
    validation_path,
    index=False
)

print("Saved:", validation_path)

print("\nFinal Stage 1 summary:")
print(f"Historical baseline rate: {baseline_probability:.1%}")
print(f"LOOCV ROC-AUC:            {model_auc:.3f}")
print(f"Player-grouped ROC-AUC:   {lopo_auc:.3f}")
print(f"Player-grouped Brier:     {lopo_brier:.3f}")
print(f"Player-grouped log loss:  {lopo_log_loss:.3f}")

Saved: ../data/stage1_validation_results.csv

Final Stage 1 summary:
Historical baseline rate: 62.5%
LOOCV ROC-AUC:            0.719
Player-grouped ROC-AUC:   0.711
Player-grouped Brier:     0.196
Player-grouped log loss:  0.556


In [24]:
# Step 16 — save leave-one-player-out predictions

stage1_predictions = stage1[
    [
        "player",
        "gsis_id",
        "season",
        "start_week",
        "team",
        "missed_game_target"
    ]
].copy()

stage1_predictions["predicted_probability"] = (
    lopo_probabilities
)

stage1_predictions["predicted_class"] = (
    lopo_predictions
)

stage1_predictions["correct"] = (
    stage1_predictions["predicted_class"]
    == stage1_predictions["missed_game_target"]
)

predictions_path = (
    DATA_DIR /
    "stage1_leave_one_player_out_predictions.csv"
)

stage1_predictions.to_csv(
    predictions_path,
    index=False
)

print("Saved:", predictions_path)
print("Rows:", len(stage1_predictions))

print("\nPrediction summary:")
print(
    stage1_predictions[
        [
            "missed_game_target",
            "predicted_probability",
            "correct"
        ]
    ]
    .describe()
    .round(3)
)

Saved: ../data/stage1_leave_one_player_out_predictions.csv
Rows: 24

Prediction summary:
       missed_game_target  predicted_probability
count              24.000                 24.000
mean                0.625                  0.581
std                 0.495                  0.240
min                 0.000                  0.207
25%                 0.000                  0.407
50%                 1.000                  0.507
75%                 1.000                  0.824
max                 1.000                  0.991
